# Experimento 007 - Treinamento real via API REST

Este notebook reproduz o escopo do Experimento 007: autenticar no TAO FTMS, validar o contexto e submeter um job real de treinamento `classification_pyt` com uma GPU e uma epoca.

A execucao mutavel fica desabilitada por padrao. Defina `RUN_EXPERIMENT_007=true` somente quando a maquina TAO estiver pronta e a GPU estiver disponivel.

## Escopo

- Health e readiness do FTMS.
- Login NGC sem exibir a chave.
- Validacao do workspace, dataset e schema de treino.
- Treinamento real com `num_epochs=1` e `num_gpus=1`.
- Polling ate estado terminal e salvamento da evidencia JSON.

Este notebook nao executa avaliacao, exportacao, TensorRT ou inferencia; essas etapas pertencem ao pipeline completo do Experimento 001.

In [1]:
import copy
import json
import os
import time
import uuid
from pathlib import Path

import requests

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://127.0.0.1:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
NETWORK_ARCH = os.getenv("TAO_REAL_NETWORK", "classification_pyt")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
RUN_EXPERIMENT = os.getenv("RUN_EXPERIMENT_007", "false").lower() == "true"
POLL_SECONDS = int(os.getenv("TAO_JOB_POLL_SECONDS", "15"))
TIMEOUT_SECONDS = int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", "3600"))
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_007_EVIDENCE", "experiment-007-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((path for path in candidates if path.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text(encoding="utf-8"))["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY:
    raise RuntimeError("NGC key not found; configure TAO_SECRETS_FILE or NGC_KEY")

http = requests.Session()
print({"base_url": TAO_BASE_URL, "org": TAO_ORG, "network": NETWORK_ARCH, "run_enabled": RUN_EXPERIMENT})

{'base_url': 'http://127.0.0.1:18090', 'org': 'getter', 'network': 'classification_pyt', 'run_enabled': True}


In [2]:
def api(method, path, payload=None, headers=None, timeout=(10, 120)):
    merged = {"Content-Type": "application/json"}
    if headers:
        merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json() if response.content else None

login = api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
health = api("GET", "/api/v2/health")
workspace = api("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schema_response = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch={NETWORK_ARCH}&action=train", headers=AUTH)
catalog = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch={NETWORK_ARCH}", headers=AUTH)

assert "readiness" in health
assert workspace.get("id") == WORKSPACE_ID
assert isinstance(schema_response, dict)
print("context=PASS datasets=", len(datasets.get("datasets", [])), "base_experiments=", len(catalog.get("experiments", [])))

context=PASS datasets= 1 base_experiments= 26


## Geracao do payload a partir do schema real

O notebook nao usa um payload fixo. Ele parte do `default` retornado pelo TAO e altera somente os parametros necessarios para o experimento.

In [3]:
train_specs = copy.deepcopy(schema_response.get("default", schema_response))
train_specs.setdefault("dataset", {}).update({"root_dir": DATASET_URI, "num_classes": int(os.getenv("TAO_NUM_CLASSES", "2"))})
train_specs.setdefault("train", {}).update({"num_gpus": 1, "num_epochs": 1})

payload = {
    "kind": "experiment",
    "name": f"experiment-007-training-{uuid.uuid4().hex[:12]}",
    "network_arch": NETWORK_ARCH,
    "encryption_key": os.getenv("TAO_ENCRYPTION_KEY", "tlt_encode"),
    "workspace": WORKSPACE_ID,
    "action": "train",
    "specs": train_specs,
    "train_dataset_uris": [DATASET_URI],
    "eval_dataset_uri": os.getenv("TAO_EVAL_DATASET_URI", DATASET_URI),
    "inference_dataset_uri": os.getenv("TAO_INFERENCE_DATASET_URI", DATASET_URI),
    "base_experiment_ids": [BASE_EXPERIMENT_ID],
}
(EVIDENCE_DIR / "train-payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print({"name": payload["name"], "epochs": train_specs["train"]["num_epochs"], "gpus": train_specs["train"]["num_gpus"], "dataset": DATASET_URI})

{'name': 'experiment-007-training-2b0fc9a64dc4', 'epochs': 1, 'gpus': 1, 'dataset': 'seaweedfs://tao-storage/data/tao-test-classification-v1'}


## Submissao e polling

A celula abaixo e mutavel. Ela exige `RUN_EXPERIMENT_007=true`; caso contrario, apenas mostra que a execucao esta protegida.

In [4]:
if not RUN_EXPERIMENT:
    print("dry-run: defina RUN_EXPERIMENT_007=true para submeter o job real")
else:
    submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    job_id = submitted["id"]
    (EVIDENCE_DIR / "train-submitted.json").write_text(json.dumps(submitted, indent=2), encoding="utf-8")
    terminal = {"Done", "Completed", "Error", "Failed", "Failure", "Canceled", "Cancelled"}
    deadline = time.monotonic() + int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", str(TIMEOUT_SECONDS)))
    while time.monotonic() < deadline:
        current = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}", headers=AUTH)
        (EVIDENCE_DIR / "train-status.json").write_text(json.dumps(current, indent=2), encoding="utf-8")
        state = current.get("status")
        print("training", state)
        if state in terminal:
            break
        time.sleep(POLL_SECONDS)
    else:
        raise TimeoutError(job_id)
    if current.get("status") not in {"Done", "Completed"}:
        raise RuntimeError(json.dumps(current, indent=2))
    print(json.dumps({"job_id": job_id, "status": current.get("status")}, indent=2))

training Pending


training Started


training Started


training Started


training Started


training Started


training Running


training Running


training Done
{
  "job_id": "0aed00ba-42c0-4af7-8d67-bbba9cf258db",
  "status": "Done"
}


In [5]:
summary = {
    "experiment": "007",
    "scope": "real training via TAO REST API",
    "base_url": TAO_BASE_URL,
    "network_arch": NETWORK_ARCH,
    "workspace_id": WORKSPACE_ID,
    "dataset_uri": DATASET_URI,
    "run_enabled": RUN_EXPERIMENT,
    "evidence_dir": str(EVIDENCE_DIR),
}
print(json.dumps(summary, indent=2))

{
  "experiment": "007",
  "scope": "real training via TAO REST API",
  "base_url": "http://127.0.0.1:18090",
  "network_arch": "classification_pyt",
  "workspace_id": "98c4bec1-88df-423e-8a15-af78881e9742",
  "dataset_uri": "seaweedfs://tao-storage/data/tao-test-classification-v1",
  "run_enabled": true,
  "evidence_dir": "experiment-007-evidence"
}
